# Exp 3 Reproducibility Check

This notebook reruns `code/03_experiment3_training.R` in an isolated scratch workspace,
compares the rerun numeric outputs against `expected/exp3_training.rds`, and compares the
regenerated figures against figures rebuilt from the expected baseline result.

In [ ]:
suppressPackageStartupMessages({
  library(survey)
  library(tidyverse)
  library(xgboost)
  library(pROC)
})

helper_candidates <- c("repro_utils.R", file.path("notebooks", "repro", "repro_utils.R"))
helper_path <- helper_candidates[file.exists(helper_candidates)][1]
if (is.na(helper_path)) {
  stop("repro_utils.R not found.")
}
source(helper_path)

baseline_result_path <- path_in_repo("expected", "exp3_training.rds")
baseline_model_df_path <- path_in_repo("data", "model_df.rds")
scratch_dir <- new_scratch("exp3_repro")

cat("Scratch workspace:", scratch_dir, "\n")
cat("Baseline result:", baseline_result_path, "\n")

In [ ]:
run_script_in_scratch(
  "code/03_experiment3_training.R",
  scratch_dir,
  c("data/model_df.rds")
)

baseline_exp3 <- readRDS(baseline_result_path)
rerun_exp3 <- readRDS(file.path(scratch_dir, "results", "exp3_training.rds"))
model_df <- readRDS(baseline_model_df_path)

In [ ]:
exp3_checks <- list(
  compare_named_numeric_list("auc", rerun_exp3$auc, baseline_exp3$auc, tol = 1e-8),
  compare_named_numeric_list("auprc", rerun_exp3$auprc, baseline_exp3$auprc, tol = 1e-8),
  compare_named_numeric_list("ci_auc", rerun_exp3$ci_auc, baseline_exp3$ci_auc, tol = 1e-8),
  compare_named_numeric_list("ci_auprc", rerun_exp3$ci_auprc, baseline_exp3$ci_auprc, tol = 1e-8),
  compare_exact_scalar("n_boot", rerun_exp3$n_boot, baseline_exp3$n_boot),
  compare_numeric_vec("pred_unwt", rerun_exp3$pred_unwt, baseline_exp3$pred_unwt, tol = 1e-8),
  compare_numeric_vec("pred_wt", rerun_exp3$pred_wt, baseline_exp3$pred_wt, tol = 1e-8),
  compare_numeric_vec("boot_auc", as.numeric(rerun_exp3$boot_auc), as.numeric(baseline_exp3$boot_auc), tol = 1e-8),
  compare_numeric_vec("boot_auprc", as.numeric(rerun_exp3$boot_auprc), as.numeric(baseline_exp3$boot_auprc), tol = 1e-8)
)

bind_rows(exp3_checks)

In [ ]:
source(path_in_repo("code", "helpers_metrics.R"))

standard_pr <- function(y, pred, n_thresholds = 200) {
  thresholds <- seq(0.01, 0.99, length.out = n_thresholds)
  map_df(thresholds, function(t) {
    pred_class <- as.numeric(pred >= t)
    tp <- sum(y == 1 & pred_class == 1)
    fp <- sum(y == 0 & pred_class == 1)
    fn <- sum(y == 1 & pred_class == 0)
    precision <- ifelse((tp + fp) > 0, tp / (tp + fp), NA)
    recall <- tp / (tp + fn)
    tibble(threshold = t, Precision = precision, Recall = recall)
  }) %>%
    filter(!is.na(Precision))
}

weighted_pr <- function(y, pred, weights, n_thresholds = 200) {
  thresholds <- seq(0.01, 0.99, length.out = n_thresholds)
  map_df(thresholds, function(t) {
    pred_class <- as.numeric(pred >= t)
    w_tp <- sum(weights[y == 1 & pred_class == 1])
    w_fp <- sum(weights[y == 0 & pred_class == 1])
    w_fn <- sum(weights[y == 1 & pred_class == 0])
    tibble(
      threshold = t,
      Precision = ifelse((w_tp + w_fp) > 0, w_tp / (w_tp + w_fp), NA),
      Recall = w_tp / (w_tp + w_fn)
    )
  }) %>%
    filter(!is.na(Precision))
}

render_exp3_figures <- function(exp3_obj, model_df, out_dir) {
  ensure_dir(out_dir)

  y <- model_df$diabetes
  weights <- model_df$WTMEC2YR
  pred_unwt <- exp3_obj$pred_unwt
  pred_wt <- exp3_obj$pred_wt

  auc_unwt_model_unwt_eval <- exp3_obj$auc$unwt_unwt
  auc_wt_model_unwt_eval <- exp3_obj$auc$wt_unwt
  auc_unwt_model_wt_eval <- exp3_obj$auc$unwt_wt
  auc_wt_model_wt_eval <- exp3_obj$auc$wt_wt

  auprc_unwt_model_unwt_eval <- exp3_obj$auprc$unwt_unwt
  auprc_wt_model_unwt_eval <- exp3_obj$auprc$wt_unwt
  auprc_unwt_model_wt_eval <- exp3_obj$auprc$unwt_wt
  auprc_wt_model_wt_eval <- exp3_obj$auprc$wt_wt

  roc_unwt_model <- roc(y, pred_unwt, quiet = TRUE)
  roc_wt_model <- roc(y, pred_wt, quiet = TRUE)
  pr_unwt_std <- standard_pr(y, pred_unwt)
  pr_wt_std <- standard_pr(y, pred_wt)
  pr_unwt_wt <- weighted_pr(y, pred_unwt, weights)
  pr_wt_wt <- weighted_pr(y, pred_wt, weights)

  roc_path <- file.path(out_dir, "fig3_roc_curves.pdf")
  pr_path <- file.path(out_dir, "fig4_pr_curves.pdf")

  pdf(roc_path, width = 12, height = 5)
  par(mfrow = c(1, 2))
  plot(roc_unwt_model, col = "blue", lwd = 2, main = "(a) Standard Evaluation", legacy.axes = TRUE)
  plot(roc_wt_model, col = "red", lwd = 2, add = TRUE)
  legend("bottomright", legend = c(sprintf("Unweighted XGB (AUC=%.3f)", auc_unwt_model_unwt_eval),
         sprintf("Weighted XGB (AUC=%.3f)", auc_wt_model_unwt_eval)), col = c("blue", "red"), lwd = 2, cex = 0.8)
  plot(roc_unwt_model, col = "blue", lwd = 2, main = "(b) Standard ROC Curves with wAUC Labels", legacy.axes = TRUE)
  plot(roc_wt_model, col = "red", lwd = 2, add = TRUE)
  legend("bottomright", legend = c(sprintf("Unweighted XGB (wAUC=%.3f)", auc_unwt_model_wt_eval),
         sprintf("Weighted XGB (wAUC=%.3f)", auc_wt_model_wt_eval)), col = c("blue", "red"), lwd = 2, cex = 0.8)
  dev.off()

  pdf(pr_path, width = 12, height = 5)
  par(mfrow = c(1, 2))
  plot(pr_unwt_std$Recall, pr_unwt_std$Precision, type = "l", col = "blue", lwd = 2,
       xlim = c(0, 1), ylim = c(0, 1), xlab = "Recall", ylab = "Precision", main = "(a) Standard Evaluation")
  lines(pr_wt_std$Recall, pr_wt_std$Precision, col = "red", lwd = 2)
  legend("topright", legend = c(sprintf("Unweighted XGB (AUPRC=%.3f)", auprc_unwt_model_unwt_eval),
         sprintf("Weighted XGB (AUPRC=%.3f)", auprc_wt_model_unwt_eval)), col = c("blue", "red"), lwd = 2, cex = 0.8)
  plot(pr_unwt_wt$Recall, pr_unwt_wt$Precision, type = "l", col = "blue", lwd = 2,
       xlim = c(0, 1), ylim = c(0, 1), xlab = "Recall", ylab = "Precision", main = "(b) Survey-Weighted Evaluation")
  lines(pr_wt_wt$Recall, pr_wt_wt$Precision, col = "red", lwd = 2)
  legend("topright", legend = c(sprintf("Unweighted XGB (wAUPRC=%.3f)", auprc_unwt_model_wt_eval),
         sprintf("Weighted XGB (wAUPRC=%.3f)", auprc_wt_model_wt_eval)), col = c("blue", "red"), lwd = 2, cex = 0.8)
  dev.off()

  c(fig3_roc_curves = roc_path, fig4_pr_curves = pr_path)
}

rerun_paths <- render_exp3_figures(rerun_exp3, model_df, file.path(scratch_dir, "out"))
baseline_paths <- render_exp3_figures(baseline_exp3, model_df, file.path(scratch_dir, "baseline_out"))

figure_checks <- list(
  compare_pdf_figure("fig3_roc_curves", rerun_paths[["fig3_roc_curves"]], baseline_paths[["fig3_roc_curves"]]),
  compare_pdf_figure("fig4_pr_curves", rerun_paths[["fig4_pr_curves"]], baseline_paths[["fig4_pr_curves"]])
)

bind_rows(figure_checks)

In [ ]:
exp3_summary <- summarize_results(c(exp3_checks, figure_checks))
exp3_summary$results